# Experiment Analysis

## Notebook Configuration

In [1]:
########## GENERAL
# Experiment directory path
EXPERIMENT_DIRPATH = "sample"

########## EXECUTION LOGS
# Function to aggregate PIT data
PIT_AGGREGATE_FUNC = "mean"

## Notebook Setup

In [2]:
# Import libraries
%matplotlib inline
import matplotlib.pyplot as plt
import os
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

## Execution Logs

In [3]:
########## UTILITIES
def load_benchmark_logs(benchmark):
    "Return a DataFrame with execution logs of the specified benchmark."
    df = pd.read_csv(os.path.join(EXPERIMENT_DIRPATH, benchmark + ".csv"))
    return df

def list_benchmarks():
    "Return a list of benchmarks in the experiment directory (CSV files)."
    return [filename.split(".")[0] for filename in os.listdir(EXPERIMENT_DIRPATH)
            if os.path.isfile(os.path.join(EXPERIMENT_DIRPATH, filename)) and filename.endswith(".csv")]

### Throughput

In [4]:
benchmarks = list_benchmarks()
fig = plt.figure(figsize=(16 * len(benchmarks), 32 * len(benchmarks)))
for (i, benchmark) in enumerate(benchmarks):
    df = load_benchmark_logs(benchmark)
    df["window"] = df.apply(lambda r: int(r["timestamp_milli"] / 1000), axis=1)
    df = df.groupby(["window"])["window"].count()
    df = df.reindex(range(0, int(df.index.max()) + 1), fill_value=0)
    ax = fig.add_subplot(len(benchmarks), 1, i + 1)
    ax.grid(alpha=0.75)
    df.plot(ax=ax, kind="bar", title="Throughput: %s benchmark" % benchmark,
            xlabel="Time (seconds)", ylabel="Calls per second",
            color="blue", grid=True, xticks=range(0, int(df.index.max()) + 1, 10))

### Point-in-Time Execution Time

In [5]:
benchmarks = list_benchmarks()
fig = plt.figure(figsize=(16 * len(benchmarks), 32 * len(benchmarks)))
for (i, benchmark) in enumerate(benchmarks):
    df = load_benchmark_logs(benchmark)
    df["window"] = df.apply(lambda r: int(r["timestamp_milli"] / 1000), axis=1)
    df = df.groupby(["window"])["exec_time_milli"].agg(PIT_AGGREGATE_FUNC)
    df = df.reindex(range(0, int(df.index.max()) + 1), fill_value=0)
    ax = fig.add_subplot(len(benchmarks), 1, i + 1)
    ax.grid(alpha=0.75)
    df.plot(ax=ax, kind="bar", title="PIT Execution Time: %s benchmark" % benchmark,
            xlabel="Time (seconds)", ylabel="%s Execution Time (milliseconds)" % PIT_AGGREGATE_FUNC,
            color="purple", grid=True, xticks=range(0, int(df.index.max()) + 1, 60))

### Execution Time Distribution

In [6]:
benchmarks = list_benchmarks()
fig = plt.figure(figsize=(16 * len(benchmarks), 32 * len(benchmarks)))
for (i, benchmark) in enumerate(benchmarks):
    df = load_benchmark_logs(benchmark)
    df["exec_time_bin"] = df.apply(lambda r: int(r["exec_time_milli"]), axis=1)
    ax = fig.add_subplot(len(benchmarks), 1, i + 1)
    ax.grid(alpha=0.75)
    ax.set_yscale("log")
    ax.set_xlim((0, df["exec_time_bin"].max()))
    df["exec_time_bin"].plot(ax=ax, kind="hist",
                             title="Execution Time Distribution: %s benchmark" % benchmark,
                             xlabel="Execution Time (milliseconds)", ylabel="Count",
                             bins=range(df["exec_time_bin"].max()),
                             grid=True, color="green")